# Класифікація діяльності людини за даними акселерометра

## Мета завдання
Розробити модель машинного навчання для класифікації 4-х видів діяльності людини:
- **Idle** (стоїть/сидить)
- **Walking** (йде)
- **Running** (біжить)
- **Stairs** (йде по сходах)


## 1. Імпорт необхідних бібліотек

In [ ]:
# Імпорт бібліотек для обробки даних
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Імпорт бібліотек для машинного навчання
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

# Налаштування візуалізації
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Ігнорування попереджень
import warnings
warnings.filterwarnings('ignore')

print('Всі бібліотеки імпортовано успішно!')

## 2. Завантаження даних

In [ ]:
# Шлях до папки з даними
data_path = 'data'

# Словник для зберігання даних
activities = {
    'idle': os.path.join(data_path, 'idle'),
    'walking': os.path.join(data_path, 'walking'),
    'running': os.path.join(data_path, 'running'),
    'stairs': os.path.join(data_path, 'stairs')
}

# Функція для завантаження всіх CSV файлів з папки
def load_activity_data(folder_path, activity_name):
    """
    Завантажує всі CSV файли з папки активності
    Повертає список DataFrame для кожного файлу
    """
    csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
    data_list = []
    
    for file_path in csv_files:
        df = pd.read_csv(file_path)
        data_list.append(df)
    
    print(f'{activity_name}: завантажено {len(data_list)} файлів')
    return data_list

# Завантажуємо дані для кожної активності
all_data = {}
for activity, folder in activities.items():
    all_data[activity] = load_activity_data(folder, activity)

print(f'\nВсього завантажено активностей: {len(all_data)}')

In [ ]:
# Переглянемо приклад даних для idle
print('Приклад даних для активності "idle":')
display(all_data['idle'][0].head(10))

print('\nСтатистика даних:')
display(all_data['idle'][0].describe())

## 3. Розрахунок часових ознак (Time Domain Features)

In [ ]:
def calculate_time_domain_features(df):
    features = {}
    
    axes = ['accelerometer_X', 'accelerometer_Y', 'accelerometer_Z']
    
    # Обчислюємо ознаки для кожної осі
    for axis in axes:
        data = df[axis]
        prefix = axis.split('_')[-1]  # X, Y або Z
        
        # Основні статистичні характеристики
        features[f'{prefix}_mean'] = data.mean()
        features[f'{prefix}_std'] = data.std()
        features[f'{prefix}_min'] = data.min()
        features[f'{prefix}_max'] = data.max()
        features[f'{prefix}_median'] = data.median()
        features[f'{prefix}_q1'] = data.quantile(0.25)
        features[f'{prefix}_q3'] = data.quantile(0.75)
        features[f'{prefix}_iqr'] = features[f'{prefix}_q3'] - features[f'{prefix}_q1']
        features[f'{prefix}_energy'] = np.sum(data ** 2)
        features[f'{prefix}_skewness'] = data.skew()
        features[f'{prefix}_kurtosis'] = data.kurtosis()
        features[f'{prefix}_mean_abs_diff'] = np.mean(np.abs(np.diff(data)))
        
        # Частота перетину нуля (zero crossing rate)
        zero_crossings = np.where(np.diff(np.sign(data)))[0]
        features[f'{prefix}_zero_crossing_rate'] = len(zero_crossings) / len(data)
    
    # Обчислюємо величину вектора прискорення (magnitude)
    magnitude = np.sqrt(df['accelerometer_X']**2 + 
                       df['accelerometer_Y']**2 + 
                       df['accelerometer_Z']**2)
    
    features['magnitude_mean'] = magnitude.mean()
    features['magnitude_std'] = magnitude.std()
    features['magnitude_min'] = magnitude.min()
    features['magnitude_max'] = magnitude.max()
    features['magnitude_median'] = magnitude.median()
    
    # Кореляція між осями
    if len(df) > 1:
        features['correlation_xy'] = df['accelerometer_X'].corr(df['accelerometer_Y'])
        features['correlation_xz'] = df['accelerometer_X'].corr(df['accelerometer_Z'])
        features['correlation_yz'] = df['accelerometer_Y'].corr(df['accelerometer_Z'])
    else:
        features['correlation_xy'] = 0
        features['correlation_xz'] = 0
        features['correlation_yz'] = 0
    
    # Коефіцієнт варіації для кожної осі
    for axis in axes:
        prefix = axis.split('_')[-1]
        mean_val = features[f'{prefix}_mean']
        if abs(mean_val) > 1e-10:  # Уникаємо ділення на нуль
            features[f'{prefix}_cv'] = features[f'{prefix}_std'] / abs(mean_val)
        else:
            features[f'{prefix}_cv'] = 0
    
    return features

print('Функцію для розрахунку ознак створено!')

In [ ]:
# Створюємо повний датасет з ознаками та мітками
def create_dataset(all_data):
    features_list = []
    labels = []
    
    activity_labels = {
        'idle': 'idle',
        'walking': 'walking',
        'running': 'running',
        'stairs': 'stairs'
    }
    
    for activity_name, data_list in all_data.items():
        print(f'Обробка активності: {activity_name} ({len(data_list)} файлів)')
        
        for df in data_list:
            # Розраховуємо ознаки
            features = calculate_time_domain_features(df)
            features_list.append(features)
            labels.append(activity_labels[activity_name])
    
    # Створюємо DataFrame
    X = pd.DataFrame(features_list)
    y = pd.Series(labels, name='activity')
    
    return X, y

# Створюємо датасет
X, y = create_dataset(all_data)

print(f'\nДатасет створено!')
print(f'Кількість семплів: {X.shape[0]}')
print(f'Кількість ознак: {X.shape[1]}')
print(f'\nРозподіл міток:')
print(y.value_counts())

In [ ]:
# Зберігаємо імена ознак для подальшого аналізу
feature_names = X.columns.tolist()
print(f'Ознаки ({len(feature_names)} шт.):')
for i, feat in enumerate(feature_names, 1):
    print(f'{i:2d}. {feat}')

# Переглядаємо перші кілька рядків датасету
print('\nПерші 5 рядків датасету:')
display(X.head())

## 4. Дослідницький аналіз даних (EDA)

In [ ]:
# Візуалізація розподілу міток класів
plt.figure(figsize=(10, 5))
y.value_counts().plot(kind='bar', color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])
plt.title('Розподіл кількості семплів за класами', fontsize=14, fontweight='bold')
plt.xlabel('Активність', fontsize=12)
plt.ylabel('Кількість семплів', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Зверніть увагу: дані незбалансовані (різна кількість семплів для кожного класу)')

In [ ]:
# Візуалізація деяких ознаок для різних класів
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Додаємо мітки до даних для візуалізації
df_viz = X.copy()
df_viz['activity'] = y

# Box plot для std кожної осі
sns.boxplot(data=df_viz, x='activity', y='X_std', ax=axes[0, 0], palette='husl')
axes[0, 0].set_title('Стандартне відхилення по осі X', fontsize=12, fontweight='bold')

sns.boxplot(data=df_viz, x='activity', y='Y_std', ax=axes[0, 1], palette='husl')
axes[0, 1].set_title('Стандартне відхилення по осі Y', fontsize=12, fontweight='bold')

sns.boxplot(data=df_viz, x='activity', y='Z_std', ax=axes[1, 0], palette='husl')
axes[1, 0].set_title('Стандартне відхилення по осі Z', fontsize=12, fontweight='bold')

sns.boxplot(data=df_viz, x='activity', y='magnitude_mean', ax=axes[1, 1], palette='husl')
axes[1, 1].set_title('Середня величина вектора прискорення', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('Box plots показують розподіл ознак для різних видів активності')

## 5. Підготовка даних для навчання моделей

In [ ]:
# Перевірка на наявність пропущених значень
print('Пропущені значення в ознаках:')
print(X.isnull().sum().sum())

# Заповнюємо пропущені значення (якщо є)
if X.isnull().sum().sum() > 0:
    X = X.fillna(0)
    print('Пропущені значення заповнено нулями')
else:
    print('Пропущених значень немає')

In [ ]:
# Розділяємо дані на тренувальну та тестову вибірки (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Зберігаємо баланс класів
)

print(f'Дані розділено!')
print(f'Тренувальна вибірка: {X_train.shape[0]} семплів')
print(f'Тестова вибірка: {X_test.shape[0]} семплів')

print('\nРозподіл міток у тренувальній вибірці:')
print(y_train.value_counts())

print('\nРозподіл міток у тестовій вибірці:')
print(y_test.value_counts())

In [ ]:
# Масштабування ознак (важливо для SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Масштабування ознак виконано!')
print(f'\nПараметри масштабування:')
print(f'Середнє (перші 5 ознак): {scaler.mean_[:5]}')
print(f'Стандартне відхилення (перші 5 ознак): {scaler.scale_[:5]}')

## 6. Навчання моделі SVM (Support Vector Machine)

### 6.1. SVM з лінійним ядром

In [ ]:
# Створюємо та навчаємо SVM з лінійним ядром
svm_linear = SVC(kernel='linear', random_state=42, C=1.0)
svm_linear.fit(X_train_scaled, y_train)

# Передбачення на тестовій вибірці
y_pred_svm_linear = svm_linear.predict(X_test_scaled)

# Обчислюємо метрики
accuracy_svm_linear = accuracy_score(y_test, y_pred_svm_linear)

print('SVM з лінійним ядром')
print(f'Accuracy: {accuracy_svm_linear:.4f}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred_svm_linear))

### 6.2. SVM з RBF ядром

In [ ]:
# Створюємо та навчаємо SVM з RBF ядром
svm_rbf = SVC(kernel='rbf', random_state=42, C=1.0, gamma='scale')
svm_rbf.fit(X_train_scaled, y_train)

# Передбачення на тестовій вибірці
y_pred_svm_rbf = svm_rbf.predict(X_test_scaled)

# Обчислюємо метрики
accuracy_svm_rbf = accuracy_score(y_test, y_pred_svm_rbf)

print('SVM з RBF ядром')
print(f'Accuracy: {accuracy_svm_rbf:.4f}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred_svm_rbf))

### 6.3. SVM з поліноміальним ядром

In [ ]:
# Створюємо та навчаємо SVM з поліноміальним ядром
svm_poly = SVC(kernel='poly', random_state=42, C=1.0, degree=3, gamma='scale')
svm_poly.fit(X_train_scaled, y_train)

# Передбачення на тестовій вибірці
y_pred_svm_poly = svm_poly.predict(X_test_scaled)

# Обчислюємо метрики
accuracy_svm_poly = accuracy_score(y_test, y_pred_svm_poly)

print('SVM з поліноміальним ядром (degree=3)')
print(f'Accuracy: {accuracy_svm_poly:.4f}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred_svm_poly))

## 7. Навчання моделі Random Forest

### 7.1. Random Forest зі 100 деревами

In [ ]:
# Створюємо та навчаємо Random Forest з 100 деревами
rf_100 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_100.fit(X_train, y_train)  # Random Forest не потребує масштабування

# Передбачення на тестовій вибірці
y_pred_rf_100 = rf_100.predict(X_test)

# Обчислюємо метрики
accuracy_rf_100 = accuracy_score(y_test, y_pred_rf_100)

print('Random Forest (100 дерев)')
print(f'Accuracy: {accuracy_rf_100:.4f}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred_rf_100))

### 7.2. Random Forest з 200 деревами

In [ ]:
# Створюємо та навчаємо Random Forest з 200 деревами
rf_200 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_200.fit(X_train, y_train)

# Передбачення на тестовій вибірці
y_pred_rf_200 = rf_200.predict(X_test)

# Обчислюємо метрики
accuracy_rf_200 = accuracy_score(y_test, y_pred_rf_200)

print('Random Forest (200 дерев)')
print(f'Accuracy: {accuracy_rf_200:.4f}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred_rf_200))

### 7.3. Random Forest з обмеженою глибиною (max_depth=10)

In [ ]:
# Random Forest з обмеженою глибиною для уникнення переобучення
rf_depth10 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_depth10.fit(X_train, y_train)

# Передбачення на тестовій вибірці
y_pred_rf_depth10 = rf_depth10.predict(X_test)

# Обчислюємо метрики
accuracy_rf_depth10 = accuracy_score(y_test, y_pred_rf_depth10)

print('Random Forest (max_depth=10)')
print(f'Accuracy: {accuracy_rf_depth10:.4f}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred_rf_depth10))

## 8. Порівняння всіх моделей

In [ ]:
# Створюємо словник з усіма моделями та їх передбаченнями
models = {
    'SVM (linear)': y_pred_svm_linear,
    'SVM (RBF)': y_pred_svm_rbf,
    'SVM (poly)': y_pred_svm_poly,
    'RF (100)': y_pred_rf_100,
    'RF (200)': y_pred_rf_200,
    'RF (depth=10)': y_pred_rf_depth10
}

# Обчислюємо метрики для кожної моделі
results = []
for model_name, y_pred in models.items():
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

# Створюємо DataFrame з результатами
results_df = pd.DataFrame(results)

print('Порівняльна таблиця всіх моделей:')
print('='*80)
display(results_df.set_index('Model'))

In [ ]:
# Візуалізація порівняння моделей
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Створюємо бар-чарти для Accuracy, Precision, F1-Score
x = np.arange(len(results_df))
width = 0.35

# Accuracy
colors = plt.cm.Set2(np.linspace(0, 1, len(results_df)))
axes[0].bar(x, results_df['Accuracy'], color=colors)
axes[0].set_xlabel('Models', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy Score', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df['Model'], rotation=45, ha='right')

# Precision
axes[1].bar(x, results_df['Precision'], color=colors)
axes[1].set_xlabel('Models', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision Score (weighted)', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(results_df['Model'], rotation=45, ha='right')

# F1-Score
axes[2].bar(x, results_df['F1-Score'], color=colors)
axes[2].set_xlabel('Models', fontsize=12)
axes[2].set_ylabel('F1-Score', fontsize=12)
axes[2].set_title('F1-Score (weighted)', fontsize=14, fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels(results_df['Model'], rotation=45, ha='right')

plt.tight_layout()
plt.show()

print('Графіки показують порівняння моделей за різними метриками')

## 9. Детальний аналіз найкращої моделі

In [ ]:
# Знаходимо модель з найвищим F1-Score
best_model_name = results_df.loc[results_df['F1-Score'].idxmax(), 'Model']
print(f'Найкраща модель за F1-Score: {best_model_name}')

# Визначаємо відповідне передбачення
best_y_pred = models[best_model_name]

# Детальний classification report
print(f'\n{"="*80}')
print(f'Детальний звіт для найкращої моделі: {best_model_name}')
print(f'{"="*80}')
print(classification_report(y_test, best_y_pred))

In [ ]:
# Матриця плутанини для найкращої моделі
cm = confusion_matrix(y_test, best_y_pred)

plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=sorted(y.unique()), 
            yticklabels=sorted(y.unique()),
            cbar_kws={'label': 'Кількість семплів'})

plt.title(f'Confusion Matrix - {best_model_name}', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

print('Матриця плутанини показує, скільки семплів було класифіковано правильно/неправильно')

## 10. Важливість ознак (для Random Forest)

In [ ]:
# Аналізуємо важливість ознак для Random Forest (100 дерев)
feature_importances = rf_100.feature_importances_

# Створюємо DataFrame з важливістю ознак
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

print('Top-15 найважливіших ознак для Random Forest:')
print('='*60)
display(importance_df.head(15))

# Візуалізація топ-15 ознак
top_15 = importance_df.head(15)

plt.figure(figsize=(12, 8))
plt.barh(range(len(top_15)), top_15['Importance'].values, color='#3498db')
plt.yticks(range(len(top_15)), top_15['Feature'].values)
plt.xlabel('Importance', fontsize=12)
plt.title('Top-15 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 11. Порівняння SVM та Random Forest на різних типах ознак

### Експеримент 1: тільки базові ознаки (mean, std, min, max)

In [ ]:
# Виділяємо базові ознаки
basic_features = [f for f in feature_names if any(f.endswith(f'_{stat}') for stat in ['mean', 'std', 'min', 'max'])]
basic_features = [f for f in basic_features if 'magnitude' not in f and 'correlation' not in f]

print(f'Базові ознаки ({len(basic_features)} шт.):')
for feat in basic_features:
    print(f'  - {feat}')

# Підмножина даних тільки з базовими ознаками
X_basic = X[basic_features]

# Розділяємо на train/test
X_basic_train, X_basic_test, y_basic_train, y_basic_test = train_test_split(
    X_basic, y, test_size=0.2, random_state=42, stratify=y
)

# Масштабування
scaler_basic = StandardScaler()
X_basic_train_scaled = scaler_basic.fit_transform(X_basic_train)
X_basic_test_scaled = scaler_basic.transform(X_basic_test)

# SVM з базовими ознаками
svm_basic = SVC(kernel='rbf', random_state=42)
svm_basic.fit(X_basic_train_scaled, y_basic_train)
y_pred_svm_basic = svm_basic.predict(X_basic_test_scaled)

# Random Forest з базовими ознаками
rf_basic = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_basic.fit(X_basic_train, y_basic_train)
y_pred_rf_basic = rf_basic.predict(X_basic_test)

print('\nResults with BASIC features only:')
print('='*60)
print('\nSVM (RBF) - Basic Features:')
print(classification_report(y_basic_test, y_pred_svm_basic))
print('Random Forest - Basic Features:')
print(classification_report(y_basic_test, y_pred_rf_basic))

### Експеримент 2: тільки ознаки magnitude

In [ ]:
# Виділяємо ознаки magnitude
magnitude_features = [f for f in feature_names if 'magnitude' in f]

print(f'Ознаки Magnitude ({len(magnitude_features)} шт.):')
for feat in magnitude_features:
    print(f'  - {feat}')

# Підмножина даних тільки з ознаками magnitude
X_magnitude = X[magnitude_features]

# Розділяємо на train/test
X_mag_train, X_mag_test, y_mag_train, y_mag_test = train_test_split(
    X_magnitude, y, test_size=0.2, random_state=42, stratify=y
)

# Масштабування
scaler_mag = StandardScaler()
X_mag_train_scaled = scaler_mag.fit_transform(X_mag_train)
X_mag_test_scaled = scaler_mag.transform(X_mag_test)

# SVM з magnitude ознаками
svm_mag = SVC(kernel='rbf', random_state=42)
svm_mag.fit(X_mag_train_scaled, y_mag_train)
y_pred_svm_mag = svm_mag.predict(X_mag_test_scaled)

# Random Forest з magnitude ознаками
rf_mag = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_mag.fit(X_mag_train, y_mag_train)
y_pred_rf_mag = rf_mag.predict(X_mag_test)

print('\nResults with MAGNITUDE features only:')
print('='*60)
print('\nSVM (RBF) - Magnitude Features:')
print(classification_report(y_mag_test, y_pred_svm_mag))
print('Random Forest - Magnitude Features:')
print(classification_report(y_mag_test, y_pred_rf_mag))

## 12. Підсумкове порівняння всіх експериментів

In [ ]:
# Підсумкова таблиця всіх експериментів
experiments = [
    ('All features - SVM (RBF)', y_test, y_pred_svm_rbf),
    ('All features - RF (100)', y_test, y_pred_rf_100),
    ('Basic features - SVM', y_basic_test, y_pred_svm_basic),
    ('Basic features - RF', y_basic_test, y_pred_rf_basic),
    ('Magnitude features - SVM', y_mag_test, y_pred_svm_mag),
    ('Magnitude features - RF', y_mag_test, y_pred_rf_mag),
]

summary_results = []
for exp_name, y_true, y_pred in experiments:
    report_dict = classification_report(y_true, y_pred, output_dict=True)
    
    summary_results.append({
        'Experiment': exp_name,
        'Accuracy': report_dict['accuracy'],
        'Precision (weighted)': report_dict['weighted avg']['precision'],
        'Recall (weighted)': report_dict['weighted avg']['recall'],
        'F1-Score (weighted)': report_dict['weighted avg']['f1-score'],
        'F1-Score (macro)': report_dict['macro avg']['f1-score']
    })

summary_df = pd.DataFrame(summary_results)

print('\nПІДСУМКОВЕ ПОРІВНЯННЯ ВСІХ ЕКСПЕРИМЕНТІВ:')
print('='*100)
display(summary_df.set_index('Experiment'))

# Знаходимо найкращий експеримент
best_exp = summary_df.loc[summary_df['F1-Score (weighted)'].idxmax()]
print(f'\nНайкращий експеримент: {best_exp.name}')
print(f'   Accuracy: {best_exp["Accuracy"]:.4f}')
print(f'   F1-Score (weighted): {best_exp["F1-Score (weighted)"]:.4f}')

In [ ]:
# Візуалізація підсумкових результатів
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x_pos = np.arange(len(summary_df))
bar_width = 0.35

# Порівняння Accuracy та F1-Score
axes[0].bar(x_pos - bar_width/2, summary_df['Accuracy'], bar_width, label='Accuracy', alpha=0.8, color='#2ecc71')
axes[0].bar(x_pos + bar_width/2, summary_df['F1-Score (weighted)'], bar_width, label='F1-Score', alpha=0.8, color='#3498db')
axes[0].set_xlabel('Experiment', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Accuracy vs F1-Score', fontsize=14, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(summary_df['Experiment'], rotation=45, ha='right', fontsize=8)
axes[0].legend()

# Порівняння Precision та Recall
axes[1].bar(x_pos - bar_width/2, summary_df['Precision (weighted)'], bar_width, label='Precision', alpha=0.8, color='#e74c3c')
axes[1].bar(x_pos + bar_width/2, summary_df['Recall (weighted)'], bar_width, label='Recall', alpha=0.8, color='#f39c12')
axes[1].set_xlabel('Experiment', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Precision vs Recall', fontsize=14, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(summary_df['Experiment'], rotation=45, ha='right', fontsize=8)
axes[1].legend()

plt.tight_layout()
plt.show()

print('Графіки показують підсумкове порівняння всіх експериментів')

In [ ]:
# Фінальний принт з найкращими результатами
print('\n' + '='*80)
print('ФІНАЛЬНІ РЕЗУЛЬТАТИ')
print('='*80)

best_exp_name = summary_df.loc[summary_df['F1-Score (weighted)'].idxmax(), 'Experiment']
best_exp_row = summary_df.loc[summary_df['F1-Score (weighted)'].idxmax()]

print(f'\nНайкраща конфігурація: {best_exp_name}')
print(f'\nМетрики:')
print(f'   • Accuracy:  {best_exp_row["Accuracy"]:.4f}')
print(f'   • Precision: {best_exp_row["Precision (weighted)"]:.4f}')
print(f'   • Recall:    {best_exp_row["Recall (weighted)"]:.4f}')
print(f'   • F1-Score:  {best_exp_row["F1-Score (weighted)"]:.4f}')

print(f'\nЗавдання виконано успішно!')
print(f'   Порівняно {len(results_df)} моделей SVM та Random Forest')
print(f'   Проведено {len(summary_df)} експериментів з різними наборами ознак')
print(f'   Використано {X.shape[0]} семплів з {X.shape[1]} ознаками')
print('='*80)